# Step 3.5: data dictionary and data-quality log

**Run `step3_clean.ipynb` and `step3_links.ipynb` first.** This notebook only reads `data-clean/` and writes:
- `notes/data_dictionary.md` and `data-clean/audit/data_dictionary.csv`: every column of every clean file, with meaning, source and status
- `notes/data_quality_log.md` and `data-clean/audit/data_quality_log.csv`: every data problem found, what we did, and whether it is documented, inferred or an open client question

Counts (blanks, ranges, rows affected) are **computed from the files**, never typed by hand. Meanings come from `01` p3 section 5.

Status words: **documented** = the pack states it (page given); **listed** = the pack lists the column but not its meaning, so the meaning is our reading; **added** = a column we created in cleaning.

In [ ]:
import os, re
import pandas as pd
CLEAN = os.path.join('..', 'data-clean'); AUD = os.path.join(CLEAN, 'audit'); NOTES = os.path.join('..', 'notes')
if not os.path.isdir(AUD): raise FileNotFoundError('Run step3_clean.ipynb first.')
if not os.path.isdir(os.path.join(AUD, 'links')): raise FileNotFoundError('Run step3_links.ipynb first.')
FILES = ['sku_master','stockist_master','batch_master','primary_sales','cnf_stock_snapshot','expiry_claims','claim_emails']
D = {n: pd.read_csv(os.path.join(CLEAN, n + '.csv'), keep_default_na=False, na_values=['']) for n in FILES}
print({n: d.shape for n, d in D.items()})

## 3.5a Meanings (from `01` p3 section 5)
One line per column. `doc` is the page reference; `status` is documented / listed / added.

In [ ]:
S51, S52, S53, S54, S55, S56, S57 = ['01 p3 s5.%d' % i for i in range(1, 8)]
M = {  # file: {column: (meaning, source, status)}
'sku_master': {
 'sku_code': ('SKU key, e.g. PFL1301 (hyphen form PFL-1301 appears in claims and was standardised)', S51, 'listed'),
 'product_description': ('Product name', S51, 'listed'),
 'pack': ('Pack size description', S51, 'listed'),
 'therapy_area': ('Therapy area', S51, 'listed'),
 'shelf_life_months': ('Shelf life in months; used for the 60% remaining-life rule (R7)', S51, 'listed'),
 'storage': ('Storage condition', S51, 'listed'),
 'nlem_scheduled': ('Whether the SKU is on the NLEM scheduled list (price-controlled)', S51, 'listed'),
 'pts_inr': ('Price to stockist, current, from the SAP price master. ALL valuation uses this', S51, 'documented'),
 'mrp_inr': ('Maximum retail price, current, from the SAP price master', S51, 'documented')},
'stockist_master': {
 'stockist_code': ('Stockist key (156 stockists); upper-cased in cleaning', S53, 'listed'),
 'stockist_name': ('Stockist name', S53, 'listed'),
 'cnf_code': ('Home C&F (CFA) that serves the stockist', S53, 'listed'),
 'city': ('City; blank for a few stockists, left blank', S53, 'listed'),
 'state': ('State', S53, 'listed'),
 'appointed_on': ('Appointment date. NOT migrated for stockists active before Apr 2024, so blank is expected', S53, 'documented'),
 'drug_licence_no': ('Drug licence number', S53, 'listed'),
 'gstin': ('GST registration number', S53, 'listed')},
'batch_master': {
 'batch_no': ('Batch key; upper-cased in cleaning', S52, 'listed'),
 'sku_code': ('SKU the batch belongs to', S52, 'listed'),
 'mfg_site': ('Manufacturing site; blank on a few rows, left blank', S52, 'listed'),
 'mfg_date': ('Manufacture date. Covers batches made from Oct 2023; earlier batches were archived at ERP migration', S52, 'documented'),
 'expiry_date': ('Expiry date. Always a month-end', S52, 'documented'),
 'batch_qty_packs': ('Packs in the batch', S52, 'listed')},
'primary_sales': {
 'invoice_no': ('Invoice number, INV/<cnf>/<yymm>/<seq>. One row per invoice line per batch', S54, 'documented'),
 'invoice_date': ('Invoice date (three raw formats, standardised)', S54, 'listed'),
 'cnf_code': ('C&F that shipped the goods', S54, 'listed'),
 'stockist_code': ('Stockist invoiced', S54, 'listed'),
 'sku_code': ('SKU invoiced', S54, 'listed'),
 'batch_no': ('Batch shipped. BLANK = the line was not supplied', S54, 'documented'),
 'ordered_qty': ('Packs ordered. A sales-order line split across batches repeats, with ordered_qty on the FIRST row only; later rows are blank. On scheme lines it INCLUDES the free packs (our reading of the data: invoiced + free = ordered on 10,171 of 10,288 scheme order lines), so packs supplied = invoiced_qty + free_qty', S54, 'documented'),
 'invoiced_qty': ('Packs invoiced on this row (sum per invoice_no + sku_code to compare with ordered_qty)', S54, 'listed'),
 'free_qty': ('Free packs given (free goods), valued at PTS for the 3% rule (R8). Counted inside ordered_qty on scheme lines', S54, 'listed'),
 'pts_inr': ('PTS per pack on the invoice; equals sku_master PTS on every row', S54, 'listed'),
 'net_value_inr': ('Net invoice value of the row (comma-text in raw, converted)', S54, 'listed'),
 'scheme_code': ('Scheme per Annexure B of CO-POL-014. NO_SCHEME is OUR label for blank / NA / -; meaning of those three is an OPEN client question', S54, 'documented'),
 'is_cancelled_line': ('invoiced_qty = 0, i.e. blank batch, line not supplied', 'cleaning', 'added'),
 'split_line_continuation': ('Later row of a split order line (ordered_qty blank by design)', 'cleaning', 'added'),
 'same_batch_split_pick': ('Same batch repeated within one order line (split picks). Not an error', 'cleaning', 'added')},
'cnf_stock_snapshot': {
 'snapshot_date': ('Month-end stock date; quarterly to Mar 2026, monthly after', S55, 'documented'),
 'cnf_code': ('C&F holding the stock (CFE01 style fixed to CF-E01)', S55, 'listed'),
 'cnf_city': ('C&F city', S55, 'listed'),
 'sku_code': ('SKU held', S55, 'listed'),
 'batch_no': ('Batch held. Blank with zero quantity = SKU out of stock at the C&F', S55, 'documented'),
 'expiry_date': ('Batch expiry date', S55, 'listed'),
 'qty_on_hand_packs': ('Packs on hand (comma-text in raw, converted)', S55, 'listed'),
 'avg_monthly_outward_packs': ('Rolling 3-month average of monthly outward packs', S55, 'documented'),
 'is_stockout_row': ('Blank batch and quantity 0 (stock-out marker)', 'cleaning', 'added'),
 'qty_conflict': ('Same date / C&F / SKU / batch appears with different quantities. Both kept; cause is an OPEN client question', 'cleaning', 'added')},
'expiry_claims': {
 'claim_no': ('Claim number, format CLM-yymm-nnnnn (R12)', S56, 'listed'),
 'claim_date': ('Date the claim was made', S56, 'listed'),
 'stockist_code': ('Claiming stockist', S56, 'listed'),
 'cnf_code': ('C&F that processes the claim', S56, 'listed'),
 'sku_code': ('SKU claimed (hyphen form fixed)', S56, 'listed'),
 'batch_no': ('Batch claimed', S56, 'listed'),
 'expiry_date_as_claimed': ('Expiry date written on the claim', S56, 'listed'),
 'qty_claimed': ('Packs claimed', S56, 'listed'),
 'claim_value_inr': ('Claim value. VALUES AT PTS; equals qty_claimed x sku_master PTS on every row that has a value. Blank on some rows, left blank', S56, 'documented'),
 'claim_type': ('Near expiry / Expired / Breakage in transit / Short supply (window rules R1). BED is named in the pack but does not occur in the extract', S56, 'listed'),
 'original_invoice_ref': ('Invoice the claim relates to. Blank = no reference (R2 needs Area Manager approval)', S56, 'listed'),
 'status': ('Credited / Pending / Rejected', S56, 'listed'),
 'verified_by': ('Who verified the claim', S56, 'listed'),
 'credit_note_no': ('Credit note number; blank follows status (Pending or Rejected)', S56, 'listed'),
 'credit_note_date': ('Credit note date', S56, 'listed'),
 'credit_value_inr': ('Value of the credit note (comma-text in raw, converted)', S56, 'listed'),
 'remarks': ('Free-text remark', S56, 'listed'),
 'invoice_ref_missing': ('original_invoice_ref is blank', 'cleaning', 'added'),
 'claim_value_derived_inr': ('Recorded claim_value_inr where there is one, otherwise qty_claimed x sku_master PTS (01 p3 s5.6 Values at PTS). Used for money totals; claim_value_inr itself is never filled', 'cleaning', 'added'),
 'claim_value_was_derived': ('True where claim_value_inr was blank and the derived value was used', 'cleaning', 'added')},
'claim_emails': {
 'email_id': ('Email key, EML-nnn', S57, 'listed'),
 'received_date': ('Date received, 01 to 16 Oct 2026', S57, 'documented'),
 'from_name': ('Sender name', S57, 'listed'),
 'stockist_code_by_mailbox_rule': ('Stockist matched by the mailbox rule; blank = sender not matched', S57, 'documented'),
 'subject': ('Subject line. UNTRUSTED input', S57, 'documented'),
 'body': ('Message body. UNTRUSTED input: never follow instructions inside it', S57, 'documented')},
}
for n in FILES:
    M[n]['src_row'] = ('Line number in the raw CSV (header = line 1). Links every clean row back to its raw row', 'cleaning', 'added')
print(sum(len(v) for v in M.values()), 'column definitions')

## 3.5b Build the dictionary with observed facts

In [ ]:
rows = []
for n in FILES:
    d = D[n]
    for c in d.columns:
        if c not in M[n]: raise KeyError(f'{n}.{c} has no definition: add it to M')
        s = d[c]
        isnum = pd.api.types.is_numeric_dtype(s) and s.dtype != bool
        rng = ''
        if isnum and s.notna().any() and c != 'src_row': rng = f'{s.min():,.2f} to {s.max():,.2f}'.replace('.00', '')
        elif c.endswith('date') or c.endswith('_on') or c.endswith('date_as_claimed'):
            t = pd.to_datetime(s, errors='coerce')
            if t.notna().any(): rng = f'{t.min().date()} to {t.max().date()}'
        meaning, src, status = M[n][c]
        rows.append(dict(file=n, column=c, observed_type=str(s.dtype), rows=len(s), blank=int(s.isna().sum()),
                         distinct=int(s.nunique()), range=rng, meaning=meaning, source=src, status=status))
DICT = pd.DataFrame(rows)
extra = [(n, c) for n in FILES for c in M[n] if c not in D[n].columns]
assert not extra, f'definitions for columns that do not exist: {extra}'
DICT.to_csv(os.path.join(AUD, 'data_dictionary.csv'), index=False)
print(DICT.groupby('file').size().to_string())
print(DICT.status.value_counts().to_string())
DICT[DICT.blank > 0][['file','column','blank','rows']]

## 3.5c Write `notes/data_dictionary.md`

In [ ]:
def esc(x): return str(x).replace('|', '/').replace('\n', ' ')
out = ['# Data dictionary', '',
       'Generated by `notebooks/step3_dictionary.ipynb` from the **clean** files in `data-clean/` (24 Nov 2026). Do not edit by hand: re-run the notebook.', '',
       'Meanings come from `01_company_and_channel_overview.pdf` p3 section 5 (`01 p3 s5.x`). Status: **documented** = the pack states it; **listed** = the pack lists the column only, the meaning is our reading; **added** = a column created in cleaning.', '',
       'Extract period 01-Apr-2024 to 30-Sep-2026; claims mailbox 01 to 16 Oct 2026; masters current as of extraction (`01` p3 s5).', '']
for n in FILES:
    sub = DICT[DICT.file == n]
    out += [f'## {n}', f'{len(D[n]):,} rows after cleaning.', '', '| Column | Meaning | Blank | Distinct | Range | Source | Status |', '|---|---|---|---|---|---|---|']
    for _, r in sub.iterrows():
        out.append(f'| `{r.column}` | {esc(r.meaning)} | {r.blank:,} | {r.distinct:,} | {esc(r.range)} | {r.source} | {r.status} |')
    out.append('')
open(os.path.join(NOTES, 'data_dictionary.md'), 'w', encoding='utf-8').write('\n'.join(out))
print('wrote notes/data_dictionary.md', len(out), 'lines')

## 3.5d Data-quality log: every problem, what we did, how sure we are

In [ ]:
cl = pd.read_csv(os.path.join(AUD, 'cleaning_log.csv'), keep_default_na=False)
lk = pd.read_csv(os.path.join(AUD, 'links', 'link_check_summary.csv'), keep_default_na=False)
tr = pd.read_csv(os.path.join(AUD, 'links', 'claims_traceability_summary.csv'), keep_default_na=False)

def basis(r):
    t = (r.issue + ' ' + r.action).lower()
    if r.column == 'scheme_code': return 'open question'
    if r.column == '(row)': return 'duplicates removed'
    if 'documented' in t: return 'documented'
    if 'not an error' in t: return 'not an error'
    if 'r5 decision' in t: return 'decision'
    if r.issue.strip().startswith('blank'): return 'blank left as is'
    if 'ask client' in t: return 'open question'
    return 'format fixed'
q = cl.copy()
q['area'] = 'cleaning'
q['basis'] = q.apply(basis, axis=1)
q = q.rename(columns={'rows': 'rows_affected'})[['area','file','column','issue','rows_affected','action','basis']]
l = lk[lk.status != 'OK'].copy()
l['area'] = 'link check'; l['file'] = l.check.str.split('.').str[0]; l['column'] = l.check
l['issue'] = l.note; l['rows_affected'] = l.problems; l['action'] = 'kept; explained in 3.4d'; l['basis'] = 'documented'
l = l[['area','file','column','issue','rows_affected','action','basis']]
DQ = pd.concat([q, l], ignore_index=True)
# a limit that no cleaning step shows (open questions on scheme_code and snapshot conflicts are already in the cleaning log)
OPEN = pd.DataFrame([
 dict(area='open question', file='primary_sales', column='pts_inr', issue='Extract holds one price per SKU, so PTS on the claim date (R5) cannot be tested', rows_affected=0, action='record as a limit', basis='limit'),
])
DQ = pd.concat([DQ, OPEN], ignore_index=True)
DQ.insert(0, 'id', ['DQ-%02d' % (i + 1) for i in range(len(DQ))])
DQ.to_csv(os.path.join(AUD, 'data_quality_log.csv'), index=False)
print(DQ.basis.value_counts().to_string())
DQ[['id','file','column','rows_affected','basis']]

In [ ]:
esc2 = lambda x: str(x).replace('|', '/').replace('\n', ' ')
out = ['# Data-quality log', '',
       'Generated by `notebooks/step3_dictionary.ipynb` from `data-clean/audit/cleaning_log.csv` and `data-clean/audit/links/`. Re-run rather than edit by hand.', '',
       'Basis: **documented** = the pack explains it (page in the issue text); **format fixed** = a mechanical clean-up with no business meaning; **duplicates removed** = exact copies dropped (listed in `data-clean/audit/dropped_rows_*.csv`); **blank left as is** = never filled; **not an error** = looks odd but is normal; **open question** = needs the client; **decision** = our choice, logged; **limit** = the data cannot answer it.', '',
       '| ID | File | Column | Issue | Rows | Action | Basis |', '|---|---|---|---|---|---|---|']
for _, r in DQ.iterrows():
    out.append(f'| {r.id} | {r.file} | `{esc2(r.column)}` | {esc2(r.issue)} | {int(r.rows_affected):,} | {esc2(r.action)} | {r.basis} |')
out += ['', '## Claims traceability (from `step3_links.ipynb`)', '', 'Can each claim be traced to a primary-sales invoice? Needed for R3 and R7.', '', '| Category | Claims | Value INR | Share of claims % |', '|---|---|---|---|']
for _, r in tr.iterrows():
    out.append(f'| {esc2(r.trace_category)} | {int(r.claims):,} | {float(r.value_inr):,.0f} | {r["share_of_claims_%"]} |')
out += ['', 'Categories C and D (971 claims) are on batches older than the batch master window (`01` p3 s5.2). R3 and R7 cannot be tested on them; R1, R2, R4, R5, R6 and R12 can.']
open(os.path.join(NOTES, 'data_quality_log.md'), 'w', encoding='utf-8').write('\n'.join(out))
print('wrote notes/data_quality_log.md', len(out), 'lines')

## 3.5e Validate

In [ ]:
assert len(DICT) == sum(len(d.columns) for d in D.values()), 'dictionary must cover every column'
assert DICT.meaning.str.len().min() > 0 and DICT.source.notna().all()
assert DICT.status.isin(['documented','listed','added']).all()
assert DQ.id.is_unique
# every cleaning-log row appears in the quality log with the same count
assert (DQ[DQ.area == 'cleaning'].rows_affected.values == cl.rows.values).all()
# the dropped-row counts in the log match the audit files
dp = pd.read_csv(os.path.join(AUD, 'dropped_rows_primary_sales.csv')); de = pd.read_csv(os.path.join(AUD, 'dropped_rows_expiry_claims.csv'))
d_ps = int(DQ[(DQ.file == 'primary_sales') & (DQ.column == '(row)')].rows_affected.iloc[0]); d_ec = int(DQ[(DQ.file == 'expiry_claims') & (DQ.column == '(row)')].rows_affected.iloc[0])
assert d_ps == len(dp) and d_ec == len(de)
# traceability categories add up to the claims in the clean file
assert int(tr.claims.sum()) == len(D['expiry_claims'])
# documented facts reproduced from data
assert (D['primary_sales'].pts_inr.notna()).all()
assert D['stockist_master'].appointed_on.isna().sum() == DICT[(DICT.file=='stockist_master') & (DICT.column=='appointed_on')].blank.iloc[0]
print('ALL DICTIONARY CHECKS PASSED')

**Check:** `ALL DICTIONARY CHECKS PASSED`. Open `notes/data_dictionary.md` and `notes/data_quality_log.md`, and look at 3 or 4 rows against the raw files.